# Emotion Recognition - Mini-Xception Training (Colab + GPU)

Bu notebook, GPU uzerinde **sadece Mini-Xception** modelini egitir ve test sonuclarini gosterir.

**Workflow:** GitHub Clone -> Kaggle Dataset -> GPU Training -> Test Sonuclari

**Onemli:** Runtime > Change runtime type > T4 GPU sectiginizden emin olun!

## 1. GPU Kontrolu

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("[WARNING] GPU bulunamadi! Runtime > Change runtime type > T4 GPU secin.")

## 2. GitHub Repo Klonlama

In [ ]:
import os

REPO_URL = "https://github.com/aysenurhepguven0/Emotion-Aware-Adaptive-Virtual-Interaction-System.git"
REPO_NAME = "Emotion-Aware-Adaptive-Virtual-Interaction-System"

%cd /content
if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}
else:
    print(f"{REPO_NAME} zaten mevcut, pull yapiliyor...")
    !cd {REPO_NAME} && git pull

%cd /content/{REPO_NAME}
print(f"[OK] Calisma dizini: {os.getcwd()}")

## 3. Bagimliliklari Yukleme

In [ ]:
!pip install -q hsemotion --no-deps
!pip install -q kagglehub

## 4. Dataset Indirme (KaggleHub)

KaggleHub ile FER2013 ve FERPlus datasetlerini indirir.

Kaggle hesabinizla otomatik authenticate olur.

In [ ]:
import kagglehub
import shutil
import os

# ----- FER2013 -----
print("FER2013 indiriliyor...")
fer_path = kagglehub.dataset_download("msambare/fer2013")
print(f"FER2013 path: {fer_path}")

# data/fer2013 klasorune kopyala
os.makedirs("data/fer2013", exist_ok=True)
!cp -r {fer_path}/* data/fer2013/
print("[OK] FER2013 yuklendi")
!ls data/fer2013/

In [ ]:
# ----- FER+ (FERPlus) -----
print("FERPlus indiriliyor...")
ferplus_path = kagglehub.dataset_download("arnabkumarroy02/ferplus")
print(f"FERPlus path: {ferplus_path}")

# data/ferplus klasorune kopyala
os.makedirs("data/ferplus", exist_ok=True)
!cp -r {ferplus_path}/* data/ferplus/
print("[OK] FERPlus yuklendi")
!ls data/ferplus/

In [ ]:
# Dataset yapisini kontrol et
import os

for ds_name in ['fer2013', 'ferplus']:
    ds_path = f'data/{ds_name}'
    if os.path.exists(ds_path):
        print(f"\n{ds_name}/")
        for split in sorted(os.listdir(ds_path)):
            split_path = os.path.join(ds_path, split)
            if os.path.isdir(split_path):
                classes = [d for d in os.listdir(split_path) if os.path.isdir(os.path.join(split_path, d))]
                total = sum(len(os.listdir(os.path.join(split_path, c))) for c in classes)
                print(f"  {split}: {total:,} images ({len(classes)} classes)")
    else:
        print(f"[ERROR] {ds_name} bulunamadi!")

## 5. Config Ayarlari (GPU icin)

In [ ]:
import torch
import config

if torch.cuda.is_available():
    config.EPOCHS = 30
    config.MODEL_CONFIGS["mini_xception"]["batch_size"] = 64
    print(f"[GPU] {config.EPOCHS} epoch, batch_size=64")
else:
    config.EPOCHS = 10
    print(f"[CPU] {config.EPOCHS} epoch")

print(f"Device: {config.DEVICE}")


## 6. Mini-Xception Model Egitimi

Secilen dataset uzerinde sadece Mini-Xception modeli egitir.

In [ ]:
# ============================================
# DATASET SECIN
# ============================================
DATASET = "ferplus"  # "fer2013", "ferplus", "rafdb", "ckplus"

!python main.py --mode train --model mini_xception --dataset {DATASET}


## 7. Test Sonuclari (Mini-Xception)

In [ ]:
import os
import config
from evaluate import evaluate_model
from models.mini_xception import get_model

MODEL_PATH = config.BEST_MODEL_PATHS["mini_xception"]
if not os.path.exists(MODEL_PATH):
    print("[ERROR] Mini-Xception modeli bulunamadi. Once egitim yapin.")
else:
    print(f"[INFO] Model yukleniyor: {MODEL_PATH}")
    model = get_model(pretrained_path=MODEL_PATH)
    results = evaluate_model(model=model, dataset_name=DATASET, split_name=f"{DATASET} Test")
    y_true = results["y_true"]
    y_pred = results["y_pred"]

    neutral_label = next(
        k for k, v in config.EMOTION_LABELS.items() if v.lower() == "neutral"
    )
    neutral_mask = y_true == neutral_label
    if neutral_mask.any():
        neutral_acc = (y_pred[neutral_mask] == neutral_label).mean() * 100
        print(f"Neutral class accuracy: {neutral_acc:.2f}% (n={neutral_mask.sum()})")
    else:
        print("[WARNING] Neutral sinifi test setinde bulunamadi.")
